In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipywidgets import interact, widgets

# подготовка данных
df_full = pd.read_csv('/content/drive/MyDrive/Учёба/8 семестр/ТИИ/mercedes_benz_sales_2020_2025.csv')
df = df_full.sample(n=50000, random_state=42).copy()

# создание колонки выручки для анализа (цена * объем продаж)
df['Revenue_USD'] = df['Base Price (USD)'] * df['Sales Volume']

# описание датасета
"""
Датасет содержит данные о продажах Mercedes-Benz.
Цель — анализ рыночных показателей (продажи, цена, выручка) по годам и моделям.

Таблица признаков:
Признак        Описание                           Единицы измерения
Model          Модель автомобиля                  Строка (категория)
Year           Год продажи                        Год (2020-2025)
Fuel_Type      Тип двигателя                      Категория
Sales_Units    Количество проданных авто          Штуки
Revenue_USD    Общая выручка                      Доллары США
Price_Avg      Средняя цена                       Доллары США
"""

# matplotlib
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

yearly_sales = df.groupby('Year')['Sales Volume'].sum()
ax1.plot(yearly_sales.index, yearly_sales.values, marker='o', color='blue')
ax1.set_title('динамика продаж по годам')
ax1.set_xlabel('год')
ax1.set_ylabel('продано единиц')

fuel_revenue = df.groupby('Fuel Type')['Revenue_USD'].sum()
ax2.bar(fuel_revenue.index, fuel_revenue.values, color='green')
ax2.set_title('выручка по типам топлива')
ax2.set_ylabel('выручка usd')

plt.tight_layout()
plt.show()

def interactive_plot(selected_year):
    data = df[df['Year'] == selected_year].groupby('Model')['Sales Volume'].sum().head(10)
    plt.figure(figsize=(10, 5))
    data.plot(kind='bar', color='orange')
    plt.title(f'продажи моделей в {selected_year} году')
    plt.ylabel('продано шт')
    plt.show()

interact(interactive_plot, selected_year=widgets.IntSlider(min=int(df['Year'].min()), max=int(df['Year'].max()), step=1, value=2020));

# pandas
# использование метода dataframe.plot (boxplot с параметром by)
df.boxplot(column='Base Price (USD)', by='Fuel Type', figsize=(10, 6))
plt.title('распределение цен по типам топлива')
plt.suptitle('')
plt.show()

# использование метода series.plot
df['Fuel Type'].value_counts().plot(kind='pie', autopct='%1.1f%%', figsize=(6, 6))
plt.title('доли типов двигателей')
plt.ylabel('')
plt.show()

# использование метода dataframe.plot (scatter)
df.plot(kind='scatter', x='Base Price (USD)', y='Sales Volume', alpha=0.3, title='связь цены и объема продаж')
plt.show()

# seaborn
sns.pairplot(df[['Sales Volume', 'Base Price (USD)', 'Revenue_USD', 'Fuel Type']], hue='Fuel Type')
plt.show()

sns.jointplot(data=df, x='Base Price (USD)', y='Revenue_USD', kind='hex')
plt.show()

plt.figure(figsize=(10, 6))
sns.violinplot(data=df, x='Fuel Type', y='Base Price (USD)')
plt.title('распределение цен по категориям топлива')
plt.show()

plt.figure(figsize=(8, 6))
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm')
plt.title('корреляция числовых признаков')
plt.show()

# scipy.stats
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(df['Base Price (USD)'], kde=True)
plt.title('гистограмма цен')

plt.subplot(1, 2, 2)
stats.probplot(df['Base Price (USD)'], dist="norm", plot=plt)
plt.title('qq-график распределения цен')
plt.show()

# plotly
fig1 = px.scatter(df.sample(2000), x="Base Price (USD)", y="Sales Volume", color="Fuel Type", title="интерактивный график цен и продаж")
fig1.show()

fig2 = make_subplots(rows=1, cols=2, subplot_titles=("гистограмма выручки", "продажи по моделям"))
fig2.add_trace(go.Histogram(x=df['Revenue_USD'], name="выручка"), row=1, col=1)
fig2.add_trace(go.Box(y=df['Sales Volume'], name="продажи"), row=1, col=2)
fig2.update_layout(title_text="суб-графики plotly")
fig2.show()

df_anim = df.groupby(['Year', 'Fuel Type'])['Sales Volume'].sum().reset_index()
fig3 = px.bar(df_anim, x="Fuel Type", y="Sales Volume", color="Fuel Type",
             animation_frame="Year", title="динамика продаж с анимацией")
fig3.show()